In [ ]:
import numpy as np

# ---------- Static data: 100 machines, 2 features (temperature, vibration) ----------
data = np.array([
    [70.6, 4.5], [67.9, 5.2], [71.5, 4.5], [71.9, 4.8],
    [66.1, 5.6], [67.4, 4.8], [70.3, 5.4], [69.4, 4.5],
    [70.0, 4.9], [68.3, 4.5], [71.8, 4.8], [71.6, 5.4],
    [70.1, 4.1], [72.3, 5.2], [70.9, 5.1], [68.3, 4.7],
    [70.7, 4.3], [68.1, 5.0], [71.8, 4.7], [69.9, 5.1],
    [69.6, 5.0], [68.6, 5.8], [72.4, 4.9], [69.7, 4.5],
    [69.1, 5.1], [69.3, 5.1], [71.1, 5.7], [70.7, 5.4],
    [70.8, 5.2], [70.9, 5.7], [74.3, 4.4], [69.2, 4.7],
    [69.0, 4.5], [68.4, 4.8], [71.2, 4.3], [72.3, 5.3],
    [69.8, 4.9], [68.3, 4.3], [68.4, 4.5], [71.3, 5.2],
    [71.5, 5.4], [71.1, 6.0], [68.7, 6.5], [70.5, 5.2],
    [70.2, 4.5], [70.4, 3.9], [71.7, 5.1], [70.4, 4.6],
    [71.4, 4.8], [70.1, 4.7], [70.6, 4.9], [71.3, 5.5],
    [67.1, 5.1], [69.4, 4.9], [69.1, 4.5], [68.7, 4.2],
    [69.4, 4.8], [73.0, 5.0], [68.3, 5.9], [71.9, 5.1],
    [66.6, 5.5], [69.3, 4.8], [70.3, 4.4], [71.2, 4.5],
    [71.4, 4.6], [71.6, 6.1], [69.3, 4.6], [69.1, 5.4],
    [71.7, 4.5], [69.6, 5.5], [67.4, 5.2], [67.7, 4.9],
    [68.2, 5.0], [71.0, 4.7], [70.3, 5.2], [71.4, 4.8],
    [69.1, 4.4], [70.3, 4.4], [71.3, 5.1], [69.4, 5.8],
    [70.9, 5.1], [68.7, 4.9], [69.3, 5.1], [69.2, 5.7],
    [67.6, 5.1], [71.0, 4.8], [69.1, 5.6], [70.0, 5.2],
    [71.0, 5.8], [70.9, 5.1], [71.3, 4.4], [69.8, 4.3],
    [69.2, 5.8], [69.8, 5.9], [66.6, 4.9], [67.1, 4.8],
    [67.4, 5.7],
    [82.5, 5.1],   # overheating machine
    [71.0, 8.9],   # normal temperature but very high vibration
    [60.2, 2.1],   # too cold and too still (machine barely running)
])
print(data.shape)          # (100, 2)

# ---------- Split ----------
X_train = data[:70]        # 70 machines, no labels needed
X_val   = data[70:]        # 30 machines, with labels
y_val   = np.array([0] * 27 + [1] * 3)   # last 3 are faulty (1 = anomaly)


# ---------- Functions ----------
def estimate_gaussian(X):
    m, n = X.shape
    mu = np.zeros(n)
    var = np.zeros(n)
    for i in range(m):
        mu = mu + X[i]
    mu = mu / m
    for i in range(m):
        var = var + (X[i] - mu) ** 2
    var = var / m
    return mu, var


def multivariate_gaussian(X, mu, var):
    p_each = (1 / np.sqrt(2 * np.pi * var)) * np.exp(-((X - mu) ** 2) / (2 * var))
    return np.prod(p_each, axis=1)


def select_threshold(y_val, p_val):
    best_epsilon = 0
    best_F1 = 0
    step_size = (max(p_val) - min(p_val)) / 1000

    for epsilon in np.arange(min(p_val), max(p_val), step_size):
        predictions = (p_val < epsilon)

        tp = np.sum((predictions == 1) & (y_val == 1))
        fp = np.sum((predictions == 1) & (y_val == 0))
        fn = np.sum((predictions == 0) & (y_val == 1))

        if tp == 0:
            continue

        prec = tp / (tp + fp)
        rec = tp / (tp + fn)
        F1 = 2 * prec * rec / (prec + rec)

        if F1 > best_F1:
            best_F1 = F1
            best_epsilon = epsilon

    return best_epsilon, best_F1


# ---------- Run the algorithm ----------
# 1. Learn what "normal" looks like from the training machines
mu, var = estimate_gaussian(X_train)
print("Mean (temp, vibration):    ", mu)
print("Variance (temp, vibration):", var)

# 2. Probability of each validation machine
p_val = multivariate_gaussian(X_val, mu, var)

# 3. Choose epsilon using the labeled validation set
epsilon, F1 = select_threshold(y_val, p_val)
print("Best epsilon:", epsilon)
print("Best F1:", F1)

# 4. Which validation machines are flagged?
flagged = p_val < epsilon
print("Flagged (1 = anomaly):", flagged.astype(int))

# 5. Check the training machines too
p_train = multivariate_gaussian(X_train, mu, var)
print("Anomalies in training set:", np.sum(p_train < epsilon))

# 6. Test a brand-new machine
new_machine = np.array([[78.0, 7.5]])
p_new = multivariate_gaussian(new_machine, mu, var)
print("New machine is", "ANOMALY" if p_new[0] < epsilon else "normal")